In [5]:
from langchain_core.prompts import ChatPromptTemplate


In [6]:
summary = """
You are a data analyst.

Your task is to create a concise but informative summary of a dataset so that another LLM can understand the dataset structure and use it for analysis or question answering.

Dataset metadata is provided below.

METADATA:
{metadata}

Write a clear dataset summary including:

1. Dataset overview
- filename
- number of rows
- number of columns

2. Columns description
For each column mention:
- column name
- possible data type (numeric / categorical / text / datetime if inferable)
- short explanation of what the column likely represents

3. Important statistics (if available)
Summarize key statistics such as:
- mean / min / max for numeric columns
- common values for categorical columns
Do not list every statistic, only the most useful ones.

4. Sample data pattern
Briefly describe patterns visible from the sample rows.

Rules:
- Keep the summary structured and concise
- Avoid repeating raw metadata
- Focus on information useful for data analysis
- Maximum 200-300 words

Return only the summary.
"""

summary_prompt = ChatPromptTemplate.from_template(summary)

insight = """
You are a senior data analyst.

Your task is to analyze the dataset summary and generate meaningful data insights.

DATASET SUMMARY:
{summary}

Generate insights about patterns, trends, anomalies, or relationships in the dataset.

Return the insights in the following JSON format.

{format_instructions}


Rules:
- Generate 5 insights.
- Focus on meaningful patterns useful for business or analysis.
- Use only information from the dataset summary.
- Avoid vague statements.
- Always return valid JSON.
"""

insight_prompt = ChatPromptTemplate.from_template(insight)

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field
from typing import List, Literal, Dict
from dotenv import load_dotenv



load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# -------------------
# Pydantic Schema
# -------------------

class Insight(BaseModel):
    title: str
    description: str
    related_columns: List[str]

    insight_type: Literal[
        "trend",
        "comparison",
        "distribution",
        "correlation",
        "anomaly",
    ]

    importance: Literal[
        "high",
        "medium",
        "low",
    ]

    suggested_visualization: Literal[
        "bar_chart",
        "line_chart",
        "histogram",
        "scatter_plot",
        "pie_chart",
    ]


class InsightsOutput(BaseModel):
    insights: List[Insight]


parser = PydanticOutputParser(pydantic_object=InsightsOutput)

format_instructions = parser.get_format_instructions()

# -------------------
# Chains
# -------------------

text_parser = StrOutputParser()

summary_chain = summary_prompt | llm | text_parser

insight_chain = insight_prompt | llm | parser


final_chain = (
    RunnablePassthrough()
    | {
        "summary": lambda x: summary_chain.invoke(
            {"metadata": x["metadata"]}
        )
    }
    | {
        "summary": lambda x: x["summary"],
        "insights": lambda x: insight_chain.invoke(
            {
                "summary": x["summary"],
                "format_instructions": format_instructions,
            }
        ),
    }
)


def generate_insights(metadata: Dict):

    res = final_chain.invoke(
        {
            "metadata": metadata
        }
    )

    return res 


In [8]:
metadata = {
    'id' : 1 ,
    'filename' : 'sales.csv',
    "columns":	["Region", "Country", "Item Type", "Sales Channel", "Order Priority", "Order Date", "Order ID", "Ship Date", "Units Sold", "Unit Price", "Unit Cost", "Total Revenue", "Total Cost", "Total Profit"],
    "num_rows" : 1330,
    "samples": [{"Region": "Europe", "Country": "Czech Republic", "Order ID": 478051030, "Item Type": "Beverages", "Ship Date": "9/29/2011", "Unit Cost": 31.79, "Order Date": "9/12/2011", "Total Cost": 151892.62, "Unit Price": 47.45, "Units Sold": 4778, "Total Profit": 74823.48, "Sales Channel": "Offline", "Total Revenue": 226716.1, "Order Priority": "C"}]
}
res = generate_insights(metadata)

In [9]:
res

{'summary': 'Dataset overview\n- Filename: sales.csv\n- Rows: 1330\n- Columns: 14\n\nColumns description\n- Region — categorical — broad geographic area (e.g., Europe)\n- Country — categorical — country within the region (e.g., Czech Republic)\n- Item Type — categorical — product category (e.g., Beverages)\n- Sales Channel — categorical — where sold (Offline/Online)\n- Order Priority — categorical — urgency label (e.g., C)\n- Order Date — datetime — date the order was placed\n- Order ID — numeric — unique identifier for the order\n- Ship Date — datetime — date the order shipped\n- Units Sold — numeric (integer) — quantity ordered\n- Unit Price — numeric — price per unit\n- Unit Cost — numeric — cost per unit\n- Total Revenue — numeric — total revenue for the order\n- Total Cost — numeric — total cost to fulfill the order\n- Total Profit — numeric — revenue minus cost\n\nImportant statistics (if available)\n- From the sample row: Units Sold = 4,778; Unit Price = 47.45; Unit Cost = 31.79

In [10]:
type(res)

dict

In [11]:
res['insights']

InsightsOutput(insights=[Insight(title='Potential profitability concentration in high-volume bulk beverage orders (sample)', description='The sample row represents a bulk order (4,778 units) of Beverages in Europe sold Offline with a unit price of 47.45 and unit cost of 31.79, yielding total revenue 226,716.10, total cost 151,892.62, and total profit 74,823.48, which implies a roughly 33% profit margin. If similar bulk orders exist in the dataset, they could be a major driver of profitability and may warrant targeted bulk-pricing or customer-segmentation strategies.', related_columns=['Units Sold', 'Unit Price', 'Unit Cost', 'Total Revenue', 'Total Cost', 'Total Profit'], insight_type='distribution', importance='high', suggested_visualization='bar_chart'), Insight(title='Europe-Beverages-Offline pattern suggests regional-channel-item clustering worth deeper analysis', description='In the sample, Region is Europe, Country is Czech Republic, Item Type is Beverages, Sales Channel is Offli

In [28]:
res['insights'] = res['insights'].model_dump()['insights']

In [29]:
res

{'summary': 'Dataset overview\n- Filename: sales.csv\n- Rows: 1330\n- Columns: 14\n\nColumns description\n- Region — categorical — broad geographic area (e.g., Europe)\n- Country — categorical — country within the region (e.g., Czech Republic)\n- Item Type — categorical — product category (e.g., Beverages)\n- Sales Channel — categorical — where sold (Offline/Online)\n- Order Priority — categorical — urgency label (e.g., C)\n- Order Date — datetime — date the order was placed\n- Order ID — numeric — unique identifier for the order\n- Ship Date — datetime — date the order shipped\n- Units Sold — numeric (integer) — quantity ordered\n- Unit Price — numeric — price per unit\n- Unit Cost — numeric — cost per unit\n- Total Revenue — numeric — total revenue for the order\n- Total Cost — numeric — total cost to fulfill the order\n- Total Profit — numeric — revenue minus cost\n\nImportant statistics (if available)\n- From the sample row: Units Sold = 4,778; Unit Price = 47.45; Unit Cost = 31.79